# 09.03 装填因子与线性探测的访存优化章节实践

本实践只依赖 Python。练习中的 Kernel（NPU 核函数）规则由 Python 模拟，Tombstone（墓碑状态）用于表示删除后仍需继续探测的槽位。请依次补全四项任务并运行每个单元格中的断言，最后再查看参考实现。

## 0. 实践目标

1. 生成与 Kernel 一致的 32 位哈希地址和线性探测序列。
2. 正确处理 EMPTY、FULL 和 TOMBSTONE。
3. 计算平均、P95、P99 和最大探测长度。
4. 计算连续 query 的多核 Tiling 参数。

In [8]:
import math
import numpy as np

EMPTY, FULL, TOMBSTONE = 0, 1, 2
U32_MASK = 0xFFFFFFFF

def hash32(key):
    value = key & U32_MASK
    value = (value * 0x9E3779B1) & U32_MASK
    value ^= value >> 16
    return value & U32_MASK

## 任务一：补全线性探测序列

In [ ]:
def probe_sequence(key, table_size, max_probe):
    # TODO：校验 table_size 为 2 的幂，再返回最多 max_probe 个回绕槽位。
    raise NotImplementedError

try:
    seq = probe_sequence(7, 8, 5)
    assert len(seq) == 5
    assert seq == [(seq[0] + i) & 7 for i in range(5)]
    print("task 1 PASSED")
except NotImplementedError:
    print("请先完成 probe_sequence")

## 任务二：实现 Tombstone 查询语义

In [ ]:
def lookup_once(table_keys, table_values, states, key, max_probe):
    # TODO：EMPTY 停止，TOMBSTONE 继续，FULL 且 key 相等时返回 value。
    # 返回 (value, hit, probes)，probes 表示实际检查的槽位数。
    raise NotImplementedError

try:
    keys = np.zeros(8, dtype=np.int32)
    values = np.zeros(8, dtype=np.int32)
    states = np.zeros(8, dtype=np.int32)
    seq = probe_sequence(7, 8, 3)
    states[seq[0]] = TOMBSTONE
    states[seq[1]] = FULL
    keys[seq[1]], values[seq[1]] = 7, 0
    assert lookup_once(keys, values, states, 7, 3) == (0, 1, 2)
    assert lookup_once(keys, values, states, 99, 8)[1] == 0
    print("task 2 PASSED")
except NotImplementedError:
    print("请先完成 lookup_once")

## 任务三：计算探测长度统计

In [ ]:
def probe_statistics(probe_counts):
    # TODO：返回 mean、p95、p99、max；输入不得为空或包含非正数。
    raise NotImplementedError

try:
    stats = probe_statistics([1, 1, 2, 2, 4])
    assert abs(stats["mean"] - 2.0) < 1e-9
    assert stats["max"] == 4
    assert stats["p99"] >= stats["p95"]
    print("task 3 PASSED")
except NotImplementedError:
    print("请先完成 probe_statistics")

## 任务四：完成多核 Tiling

In [ ]:
def compute_tiling(query_count, available_cores, query_tile=128):
    # TODO：返回 block_num、queries_per_core、tiles_per_core 和 tail_queries。
    # tail_queries 表示最后一个实际活跃核的 valid_len，而不是最后一个启动核的工作量。
    raise NotImplementedError

try:
    assert compute_tiling(1, 20) == (1, 1, 1, 1)
    assert compute_tiling(1000, 20) == (20, 50, 1, 50)
    assert compute_tiling(257, 20) == (20, 13, 1, 10)
    # ceil 分配可能让末尾启动核空闲；尾块仍应定位到最后一个实际活跃核。
    assert compute_tiling(101, 20) == (20, 6, 1, 5)
    assert compute_tiling(6, 5) == (5, 2, 1, 2)
    print("task 4 PASSED")
except NotImplementedError:
    print("请先完成 compute_tiling")

## 查看参考答案

完成四项任务后运行下面的单元格。`cat` 只显示答案文件，不会替你执行答案。

In [ ]:
!cat answer/09.03_chapter_practice/practice_solution.py

## 验收清单

- 四项任务全部通过。
- 能解释为什么墓碑不能让查询提前结束。
- 能区分命中 value 为 0 与未命中。
- 能说明 P99 比平均值更容易暴露长探测 query。
- 能把 `queries_per_core`、`QUERY_TILE` 和尾块 `valid_len` 对应起来。